# Chapter 16 — FastAPI Streaming Endpoint

You've trained and aligned your model — now make it available to users. Deployment bridges the gap between a trained model file and a product people can use. This chapter covers serving with FastAPI, token streaming, building a chat interface, and production concerns.

### Glossary / Glossário

• FastAPI — modern Python web framework for building high-performance APIs. / Framework web Python moderno para construir APIs de alta performance.
• SSE — Server-Sent Events, protocol for streaming data from server to client over HTTP. / Eventos enviados pelo servidor, protocolo para streaming de dados do servidor ao cliente via HTTP.
• streaming — sending generated tokens to the client as they are produced, rather than waiting for completion. / Enviar tokens gerados ao cliente conforme são produzidos, sem esperar conclusão.
• JWT — JSON Web Token, compact token format for stateless authentication. / JSON Web Token, formato compacto de token para autenticação stateless.
• rate limiting — restricting the number of API requests a client can make in a time period. / Restrição do número de requisições que um cliente pode fazer em um período.
• TTFT — Time To First Token, latency between sending a request and receiving the first generated token. / Tempo até o primeiro token, latência entre enviar requisição e receber o primeiro token gerado.

In [ ]:
import json
import time

print("=== Server-Sent Events (SSE) Streaming ===\n")

def simulate_generation(prompt):
    tokens = ["Once", " upon", " a", " time", ",", " in",
              " a", " land", " far", " away", "..."]  # List of generated tokens - simulating model output one at a time
    for token in tokens:
        yield token

def format_sse(prompt):
    for token in simulate_generation(prompt):
        yield f"data: {json.dumps({'token': token, 'finish_reason': None})}"
    yield f"data: {json.dumps({'token': '', 'finish_reason': 'stop'})}"
    yield "data: [DONE]"

full_text = ""  # Accumulated text reconstructed from individual SSE events
for event in format_sse("Tell me a story"):
    print(event)
    if event.startswith("data: {"):
        data = json.loads(event[6:])
        if data.get("token"):
            full_text += data["token"]

print(f"\nReconstructed: \"{full_text}\"")
print(f"Each token streams as an SSE event immediately after generation.")

# --- Aha demo: Progressive token arrival with timing ---
print("\n=== Aha! Streaming vs. Waiting ===\n")
print("Simulating tokens arriving with latency (50ms each):\n")

tokens = ["The", " quick", " brown", " fox", " jumps",
          " over", " the", " lazy", " dog", "."]  # List of generated tokens - simulating model output one at a time

accumulated = ""  # Accumulated text reconstructed from individual SSE events
start = time.time()

for i, token in enumerate(tokens):
    time.sleep(0.05)  # Simulate 50ms generation latency per token
    accumulated += token
    elapsed = time.time() - start
    print(f"  t={elapsed:.2f}s | token {i+1:>2d}: {repr(token):>8s} | text so far: \"{accumulated}\"")

total_time = time.time() - start
print(f"\nStreaming: user sees first token at ~0.05s, full text at {total_time:.2f}s")
print(f"Without streaming: user waits {total_time:.2f}s seeing nothing, then gets everything at once.")
print(f"-> Streaming gives the same total time but much better perceived latency!")

In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import asyncio

app = FastAPI()

async def generate_stream(prompt: str):
    for token in model.generate(prompt, stream=True):
        yield f"data: {json.dumps({'token': token})}\n\n"
        await asyncio.sleep(0)
    yield "data: [DONE]\n\n"

@app.post("/v1/chat/completions")
async def chat(request: ChatRequest):
    return StreamingResponse(
        generate_stream(request.messages),
        media_type="text/event-stream",
    )

---

**Course:** Aprenda LLM | **Chapter 16**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.